# Phase 9.6: Diverse Dataset Injection (Training-Free)

**Physics-Based Knowledge Injection via Thermodynamic Settling**

This notebook implements TRUE FLUX learning — no backpropagation required.

## Philosophy

Traditional ML: `loss.backward()` → gradient → weight update  
**FLUX Physics**: `CSE.encode()` → field coordinates → mass injection → `field.settle()`

The field learns by **accumulating evidence** at semantic locations:
- More evidence → higher mass → stronger gravitational pull
- Related concepts cluster naturally via wave interference
- Contradictions create repulsive (negative mass) zones

## Datasets (Diverse Knowledge Injection)

| Domain | Dataset | Why |
|--------|---------|-----|
| **Assistancy** | OpenAssistant | Multi-turn helpful responses |
| **Dialogue** | Dolly-15k | Natural conversation patterns |
| **Reasoning** | GSM8K | Math step-by-step reasoning |
| **Tool Use** | ToolBench | API/function calling patterns |
| **Code** | CodeAlpaca | Programming knowledge |
| **Facts** | TriviaQA | Factual knowledge grounding |

## Expected Runtime
- ~15-30 min on T4 GPU (full injection)
- ~45 min with optional decoder fine-tuning

---

## Cell 1: Clone/Pull FLUX Repository

In [ ]:
%%time
import os
from pathlib import Path

REPO_URL = 'https://github.com/Unseengap/FLUX.git'
ROOT = Path('/kaggle/working/FLUX') if Path('/kaggle').exists() else Path('/content/FLUX')

if ROOT.exists():
    print('  Pulling latest...')
    !cd {ROOT} && git pull
else:
    print('  Cloning repo...')
    !git clone {REPO_URL} {ROOT}

os.chdir(ROOT)
print(f'  Working dir: {os.getcwd()}')

# Install dependencies
!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt
!pip install -q huggingface_hub datasets transformers

print('  ✓ Dependencies installed')

## Cell 2: Setup Paths & Logger

In [ ]:
import sys
from pathlib import Path

ROOT = Path('/kaggle/working/FLUX') if Path('/kaggle').exists() else Path('/content/FLUX')
ROOT_DIR = ROOT
PHASES_DIR = ROOT / 'phases'
CHECKPOINT_DIR = ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Add all phase paths
for i in range(1, 10):
    p = PHASES_DIR / f'phase{i}'
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Add phase subdirs
for subdir in ['phase8_8', 'phase8_9', 'phase9']:
    p = PHASES_DIR / subdir
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        print(f'  ✓ {subdir} found')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from flux_utils import PhaseLogger, get_device

log = PhaseLogger(phase=9)  # Log to phase9.log
log.separator("Phase 9.6: Diverse Dataset Injection")

print('  ✓ Paths configured')

## Cell 3: Hardware & Secrets

In [ ]:
import torch
import os

log.cell_start("Cell 3 — Hardware & Secrets")

device = get_device()
print(f'  Device: {device}')

if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU: {gpu_name} ({vram:.1f} GB)')

# Load secrets (Kaggle → Colab → env fallback)
hf_token = None

# Try Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    print('  ✓ Kaggle secrets loaded')
except:
    pass

# Try Colab secrets
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        print('  ✓ Colab secrets loaded')
    except:
        pass

# Fall back to env vars
if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')
    if hf_token:
        print('  ✓ Environment variables loaded')

# Clean and set token
if hf_token:
    hf_token = hf_token.strip()
    os.environ['HF_TOKEN'] = hf_token
    print('  ✓ HF_TOKEN set')
else:
    print('  ⚠ HF_TOKEN not found (will try anonymous download)')

log.cell_end("Cell 3 — Hardware & Secrets", "PASS")

## Cell 4: Download & Load FLUX Model

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path

log.cell_start("Cell 4 — Download & Load FLUX Model")

# Priority: Flux-X-enriched.flx > Flux-X-complete.flx > Flux-beta.flx
FLX_OPTIONS = [
    ('Flux-X-enriched.flx', 'checkpoints/Flux-X-enriched.flx'),  # Already has GPT-2 knowledge
    ('Flux-X-complete.flx', 'checkpoints/Flux-X-complete.flx'),
    ('Flux-X.flx', 'checkpoints/Flux-X.flx'),
    ('Flux-beta.flx', 'checkpoints/Flux-beta.flx'),
]

flx_path = None

# Check local first
for name, _ in FLX_OPTIONS:
    local_path = CHECKPOINT_DIR / name
    if local_path.exists():
        flx_path = local_path
        size_mb = flx_path.stat().st_size / 1e6
        print(f'  ✓ Found local {name} ({size_mb:.1f} MB)')
        break

# Download if not found
if flx_path is None:
    for name, hf_path in FLX_OPTIONS:
        print(f'  Trying to download {name}...')
        try:
            downloaded = hf_hub_download(
                repo_id='UnseenGAP/FLUX',
                filename=hf_path,
                local_dir=str(ROOT_DIR),
                token=hf_token,
            )
            flx_path = CHECKPOINT_DIR / name
            size_mb = Path(downloaded).stat().st_size / 1e6
            print(f'  ✓ Downloaded {name} ({size_mb:.1f} MB)')
            break
        except Exception as e:
            print(f'  ⚠ {name} not available: {e}')
            continue

if flx_path is None:
    raise RuntimeError("Cannot proceed without FLUX checkpoint")

# Load the model
try:
    from flx_loader import verify_flx, get_flx_info, load_flux_from_flx
    print('  ✓ flx_loader imported')
except ImportError as e:
    print(f'  ⚠ Import error: {e}')
    raise

# Verify the archive
success, msg = verify_flx(flx_path)
print(f'  Verification: {msg}')

if not success:
    raise RuntimeError(f"FLX verification failed: {msg}")

# Show info
info = get_flx_info(flx_path)
print(f'  Version: {info["version"]}')

# Load the full model
model = load_flux_from_flx(
    path=flx_path,
    device=device,
    verbose=True,
    auto_download=False,
)

# Quick stats
stats = model.get_stats()
log.metric("Total params", f"{stats.total_params:,}")
log.metric("Field energy", f"{stats.field_energy:.2f}")

log.cell_end("Cell 4 — Download & Load FLUX Model", "PASS")

## Cell 5: Load Diverse Datasets

We load multiple datasets covering different capabilities:

| Dataset | Domain | Size | Key Features |
|---------|--------|------|--------------|
| OpenAssistant | Assistancy | ~10k | Multi-turn, helpful |
| Dolly-15k | Dialogue | 15k | Natural conversation |
| GSM8K | Reasoning | 8.5k | Math + chain-of-thought |
| Capybara | Multi-turn | 16k | Complex conversations |
| Alpaca-Code | Code | 20k | Programming tasks |

In [ ]:
log.cell_start("Cell 5 — Load Diverse Datasets")

from datasets import load_dataset
import random

print("\n  Loading diverse datasets...")
print("  " + "="*50)

datasets_loaded = {}

# ─────────────────────────────────────────────
# 1. OpenAssistant Conversations (Assistancy)
# ─────────────────────────────────────────────
print("\n  [1/5] OpenAssistant Conversations...")
try:
    oasst = load_dataset("OpenAssistant/oasst1", split="train")
    # Extract human-assistant pairs
    oasst_texts = []
    for item in oasst:
        if item.get('text'):
            oasst_texts.append(item['text'])
    datasets_loaded['assistancy'] = oasst_texts[:5000]  # Limit for speed
    print(f"       ✓ Loaded {len(datasets_loaded['assistancy']):,} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    datasets_loaded['assistancy'] = []

# ─────────────────────────────────────────────
# 2. Dolly-15k (Natural Dialogue)
# ─────────────────────────────────────────────
print("\n  [2/5] Dolly-15k Dialogue...")
try:
    dolly = load_dataset("databricks/databricks-dolly-15k", split="train")
    dolly_texts = []
    for item in dolly:
        # Combine instruction + context + response
        text = f"User: {item['instruction']}"
        if item.get('context'):
            text += f"\nContext: {item['context']}"
        text += f"\nAssistant: {item['response']}"
        dolly_texts.append(text)
    datasets_loaded['dialogue'] = dolly_texts[:5000]
    print(f"       ✓ Loaded {len(datasets_loaded['dialogue']):,} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    datasets_loaded['dialogue'] = []

# ─────────────────────────────────────────────
# 3. GSM8K (Math Reasoning)
# ─────────────────────────────────────────────
print("\n  [3/5] GSM8K Math Reasoning...")
try:
    gsm8k = load_dataset("gsm8k", "main", split="train")
    gsm_texts = []
    for item in gsm8k:
        # Include step-by-step reasoning
        text = f"Problem: {item['question']}\n\nSolution: {item['answer']}"
        gsm_texts.append(text)
    datasets_loaded['reasoning'] = gsm_texts  # All 7.5k
    print(f"       ✓ Loaded {len(datasets_loaded['reasoning']):,} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    datasets_loaded['reasoning'] = []

# ─────────────────────────────────────────────
# 4. Capybara (Multi-turn Conversations)
# ─────────────────────────────────────────────
print("\n  [4/5] Capybara Multi-turn...")
try:
    capybara = load_dataset("LDJnr/Capybara", split="train")
    capy_texts = []
    for item in capybara:
        if 'conversation' in item:
            conv = item['conversation']
            if isinstance(conv, list):
                text = "\n".join([f"{turn.get('role', 'user')}: {turn.get('content', '')}" 
                                  for turn in conv if isinstance(turn, dict)])
                capy_texts.append(text)
    datasets_loaded['multiturn'] = capy_texts[:5000]
    print(f"       ✓ Loaded {len(datasets_loaded['multiturn']):,} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    datasets_loaded['multiturn'] = []

# ─────────────────────────────────────────────
# 5. Code Alpaca (Programming)
# ─────────────────────────────────────────────
print("\n  [5/5] Code Alpaca Programming...")
try:
    code = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
    code_texts = []
    for item in code:
        text = f"Task: {item.get('instruction', '')}"
        if item.get('input'):
            text += f"\nInput: {item['input']}"
        text += f"\nCode: {item.get('output', '')}"
        code_texts.append(text)
    datasets_loaded['code'] = code_texts[:5000]
    print(f"       ✓ Loaded {len(datasets_loaded['code']):,} samples")
except Exception as e:
    print(f"       ⚠ Failed: {e}")
    datasets_loaded['code'] = []

# ─────────────────────────────────────────────
# Summary
# ─────────────────────────────────────────────
print("\n  " + "="*50)
print("  DATASET SUMMARY:")
total_samples = 0
for domain, texts in datasets_loaded.items():
    count = len(texts)
    total_samples += count
    print(f"    {domain:12s}: {count:,} samples")
print(f"  {'─'*30}")
print(f"    {'TOTAL':12s}: {total_samples:,} samples")

log.metric("Total samples", f"{total_samples:,}")
log.metric("Domains", len([d for d in datasets_loaded if datasets_loaded[d]]))

log.cell_end("Cell 5 — Load Diverse Datasets", "PASS")

## Cell 6: Training-Free Injection Framework

**The FLUX Physics Learning Loop:**

```
for text in dataset:
    wave = CSE.encode(text)           # Deterministic wave encoding
    location = hash_coords(wave)       # Map to 3D field coordinates
    field.state[location] += wave      # Inject semantic pattern
    field.mass[location] += evidence   # Accumulate evidence
    field.settle()                     # Thermodynamic equilibrium
```

**No gradients. No backprop. Just physics.**

The field naturally organizes:
- Similar concepts cluster (wave interference)
- High-evidence locations become attractors (gravity)
- The decoder later reads this enriched topology

In [ ]:
log.cell_start("Cell 6 — Training-Free Injection Framework")

import hashlib
import torch
import time

# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────

INJECTION_SCALE = 0.1      # Per-sample injection strength
MASS_PER_SAMPLE = 1.0      # Evidence per sample
SETTLE_EVERY = 500         # Thermodynamic settle every N samples
SETTLE_STEPS = 3           # Settling iterations
MAX_TEXT_LENGTH = 512      # Truncate long texts
INJECTION_RADIUS = 1       # Local neighborhood size

print("\n  Training-Free Injection Framework")
print("  " + "="*50)
print(f"    Injection scale: {INJECTION_SCALE}")
print(f"    Mass per sample: {MASS_PER_SAMPLE}")
print(f"    Settle every: {SETTLE_EVERY} samples")
print(f"    Max text length: {MAX_TEXT_LENGTH} chars")

# ─────────────────────────────────────────────
# Core functions
# ─────────────────────────────────────────────

def text_to_field_coords(text: str, cse, field, use_hash: bool = True):
    """
    Convert text → field coordinates using CSE + hash.
    
    Returns:
        (h, w, d): Field coordinates
        wave_features: [features] semantic features to inject
    """
    H, W, D = field.h, field.w, field.d
    
    # Encode with CSE
    wave = cse.encode(text[:MAX_TEXT_LENGTH])
    
    # Get wave vector
    if hasattr(wave, 'full'):
        wave_vec = wave.full.mean(dim=0)  # Average over sequence
    elif hasattr(wave, 'semantic'):
        wave_vec = wave.semantic.mean(dim=0)
    else:
        wave_vec = wave.mean(dim=0) if wave.dim() > 1 else wave
    
    wave_vec = wave_vec.cpu()
    
    if use_hash:
        # Hash for unique spread, CSE offset for semantic clustering
        h_hash = hashlib.md5(text.encode()).hexdigest()
        base_h = int(h_hash[:8], 16) % H
        base_w = int(h_hash[8:16], 16) % W
        base_d = int(h_hash[16:24], 16) % D
        
        # Small CSE offset for semantic neighborhoods
        off_h = int((wave_vec[0].item() * 100) % 5) - 2
        off_w = int((wave_vec[100 % len(wave_vec)].item() * 100) % 5) - 2
        off_d = int((wave_vec[200 % len(wave_vec)].item() * 100) % 5) - 2
        
        h = max(0, min(H - 1, base_h + off_h))
        w = max(0, min(W - 1, base_w + off_w))
        d = max(0, min(D - 1, base_d + off_d))
    else:
        # Pure CSE-based coordinates
        h = int((wave_vec[0].item() + 1) * (H - 1) / 2) % H
        w = int((wave_vec[100 % len(wave_vec)].item() + 1) * (W - 1) / 2) % W
        d = int((wave_vec[200 % len(wave_vec)].item() + 1) * (D - 1) / 2) % D
    
    # Prepare injection features (project to field dimension)
    if field.features == 512 and len(wave_vec) >= 256:
        # Use semantic portion if available
        features = torch.zeros(512)
        features[:min(512, len(wave_vec))] = wave_vec[:min(512, len(wave_vec))]
    else:
        features = torch.zeros(field.features)
        copy_len = min(len(wave_vec), field.features)
        features[:copy_len] = wave_vec[:copy_len]
    
    return (h, w, d), features


def inject_into_field(field, coords, features, scale=0.1, mass=1.0, radius=1):
    """
    Inject semantic features into field at given coordinates.
    """
    h, w, d = coords
    H, W, D = field.h, field.w, field.d
    
    # Get neighborhood slices
    h_slice = slice(max(0, h - radius), min(H, h + radius + 1))
    w_slice = slice(max(0, w - radius), min(W, w + radius + 1))
    d_slice = slice(max(0, d - radius), min(D, d + radius + 1))
    
    # Inject features (additive)
    injection = features.to(field.state.device)
    field.state[h_slice, w_slice, d_slice] += injection * scale
    
    # Add mass at center (evidence accumulation)
    field.mass[h, w, d] += mass
    
    # Lower energy (more stable)
    field.energy[h_slice, w_slice, d_slice] *= 0.98


def thermodynamic_settle(field, steps=3):
    """
    Settle field to thermodynamic equilibrium.
    Simulates diffusion + energy minimization.
    """
    with torch.no_grad():
        for _ in range(steps):
            # Simple diffusion: average with neighbors
            # This is a basic approximation of thermodynamic settling
            padded = torch.nn.functional.pad(field.state, (0,0,1,1,1,1,1,1), mode='replicate')
            
            # 6-neighbor average (3D)
            neighbors = (
                padded[:-2, 1:-1, 1:-1, :] +  # h-1
                padded[2:, 1:-1, 1:-1, :] +   # h+1
                padded[1:-1, :-2, 1:-1, :] +  # w-1
                padded[1:-1, 2:, 1:-1, :] +   # w+1
                padded[1:-1, 1:-1, :-2, :] +  # d-1
                padded[1:-1, 1:-1, 2:, :]     # d+1
            ) / 6
            
            # Blend current state with neighbor average (0.1 diffusion rate)
            field.state = 0.95 * field.state + 0.05 * neighbors
            
            # Energy decay
            field.energy *= 0.99


print("  ✓ Injection framework defined")
print(f"    • text_to_field_coords(text, cse, field)")
print(f"    • inject_into_field(field, coords, features)")
print(f"    • thermodynamic_settle(field, steps)")

log.cell_end("Cell 6 — Training-Free Injection Framework", "PASS")

## Cell 7: Inject All Datasets into Field

This is where the magic happens — streaming all diverse datasets through the physics-based learning loop.

In [ ]:
log.cell_start("Cell 7 — Inject All Datasets into Field")

import time
from collections import defaultdict

print("\n  Starting Training-Free Dataset Injection")
print("  " + "="*50)
print("  NO GRADIENTS. NO BACKPROP. JUST PHYSICS.")
print("  " + "="*50)

# ─────────────────────────────────────────────
# Pre-injection field stats
# ─────────────────────────────────────────────

pre_stats = {
    'mass_sum': model.field.mass.sum().item(),
    'state_norm': model.field.state.norm().item(),
    'energy_mean': model.field.energy.mean().item(),
}
print(f"\n  Pre-injection field stats:")
print(f"    Mass sum: {pre_stats['mass_sum']:,.0f}")
print(f"    State norm: {pre_stats['state_norm']:.2f}")
print(f"    Energy mean: {pre_stats['energy_mean']:.4f}")

# ─────────────────────────────────────────────
# Injection loop
# ─────────────────────────────────────────────

model.cse.eval()
injection_stats = defaultdict(lambda: {'count': 0, 'failed': 0})
location_histogram = torch.zeros(model.field.h, model.field.w, model.field.d)

total_injected = 0
total_failed = 0
start_time = time.time()

# Domain-specific injection scales (some domains need stronger signal)
DOMAIN_SCALES = {
    'assistancy': 0.15,   # Helpful patterns
    'dialogue': 0.12,     # Natural conversation
    'reasoning': 0.20,    # Critical for CoT
    'multiturn': 0.12,    # Context handling
    'code': 0.15,         # Programming patterns
}

print(f"\n  Injecting datasets...")

for domain, texts in datasets_loaded.items():
    if not texts:
        continue
    
    domain_scale = DOMAIN_SCALES.get(domain, INJECTION_SCALE)
    print(f"\n  [{domain.upper()}] {len(texts):,} samples (scale={domain_scale})")
    
    domain_start = time.time()
    
    for i, text in enumerate(texts):
        if not text or len(text.strip()) < 10:
            injection_stats[domain]['failed'] += 1
            total_failed += 1
            continue
        
        try:
            # Convert text → field coordinates + features
            coords, features = text_to_field_coords(text, model.cse, model.field)
            
            # Inject into field
            inject_into_field(
                model.field, 
                coords, 
                features, 
                scale=domain_scale, 
                mass=MASS_PER_SAMPLE,
                radius=INJECTION_RADIUS
            )
            
            # Track location
            h, w, d = coords
            location_histogram[h, w, d] += 1
            
            injection_stats[domain]['count'] += 1
            total_injected += 1
            
        except Exception as e:
            injection_stats[domain]['failed'] += 1
            total_failed += 1
            continue
        
        # Thermodynamic settling every N samples
        if (total_injected + 1) % SETTLE_EVERY == 0:
            thermodynamic_settle(model.field, steps=SETTLE_STEPS)
            elapsed = time.time() - start_time
            rate = total_injected / elapsed
            print(f"    ... {total_injected:,} injected ({rate:.0f}/sec) — settling...")
    
    domain_elapsed = time.time() - domain_start
    print(f"    ✓ {injection_stats[domain]['count']:,} injected in {domain_elapsed:.1f}s")

# Final settling pass
print(f"\n  Final thermodynamic settling...")
thermodynamic_settle(model.field, steps=10)

total_time = time.time() - start_time

# ─────────────────────────────────────────────
# Post-injection stats
# ─────────────────────────────────────────────

post_stats = {
    'mass_sum': model.field.mass.sum().item(),
    'state_norm': model.field.state.norm().item(),
    'energy_mean': model.field.energy.mean().item(),
}

unique_locations = (location_histogram > 0).sum().item()
max_overlap = location_histogram.max().item()

print(f"\n  " + "="*50)
print(f"  ✓ INJECTION COMPLETE")
print(f"  " + "="*50)
print(f"\n  Injection summary:")
print(f"    Total injected: {total_injected:,}")
print(f"    Total failed: {total_failed:,}")
print(f"    Total time: {total_time:.1f}s ({total_injected/total_time:.0f} samples/sec)")
print(f"    Unique locations: {unique_locations:,}")
print(f"    Max samples per location: {max_overlap:.0f}")

print(f"\n  Field changes:")
print(f"    Mass: {pre_stats['mass_sum']:,.0f} → {post_stats['mass_sum']:,.0f} (+{post_stats['mass_sum']-pre_stats['mass_sum']:,.0f})")
print(f"    State norm: {pre_stats['state_norm']:.2f} → {post_stats['state_norm']:.2f}")
print(f"    Energy: {pre_stats['energy_mean']:.4f} → {post_stats['energy_mean']:.4f}")

print(f"\n  Per-domain breakdown:")
for domain, stats in injection_stats.items():
    print(f"    {domain:12s}: {stats['count']:,} injected, {stats['failed']:,} failed")

log.metric("Total injected", f"{total_injected:,}")
log.metric("Unique locations", f"{unique_locations:,}")
log.metric("Injection time", f"{total_time:.1f}s")
log.metric("Field mass delta", f"+{post_stats['mass_sum']-pre_stats['mass_sum']:,.0f}")

log.success("Training-free injection complete!")
log.cell_end("Cell 7 — Inject All Datasets into Field", "PASS")

## Cell 8: Post-Injection Generation Test

Let's see if the enriched field improves generation quality.

In [ ]:
log.cell_start("Cell 8 — Post-Injection Generation Test")

print("\n  Post-Injection Generation Test")
print("  " + "="*50)

# Test prompts covering injected domains
test_prompts = {
    'general': [
        "Hello, how are you?",
        "The sky is",
        "Once upon a time",
    ],
    'assistancy': [
        "Can you help me with",
        "Please explain how to",
        "What is the best way to",
    ],
    'reasoning': [
        "If I have 5 apples and give away 2,",
        "To solve this problem, first",
        "Step 1:",
    ],
    'dialogue': [
        "User: Hello\nAssistant:",
        "Question: What is AI?\nAnswer:",
    ],
    'code': [
        "def fibonacci(n):",
        "# Function to sort a list",
        "import pandas as pd",
    ],
}

model.eval()
with torch.no_grad():
    for category, prompts in test_prompts.items():
        print(f"\n  [{category.upper()}]")
        for prompt in prompts:
            try:
                response = model.generate(prompt, max_length=40, temperature=0.7)
                text = response.text if hasattr(response, 'text') else str(response)
                # Truncate for display
                display_text = text[:60] + "..." if len(text) > 60 else text
                print(f'    "{prompt[:30]}..." → {display_text}')
            except Exception as e:
                print(f'    "{prompt[:30]}..." → ERROR: {e}')

print("\n  " + "="*50)
print("  Note: Even with enriched field, decoder still needs")
print("  fine-tuning to properly 'read' the new knowledge.")
print("  Run Cell 9 for optional decoder training.")

log.cell_end("Cell 8 — Post-Injection Generation Test", "PASS")

## Cell 9: Optional — Decoder Fine-Tuning

While the field now contains diverse knowledge, the decoder was trained on a different field topology. This optional cell fine-tunes the decoder to read the enriched field.

**Skip this if you just want the enriched field checkpoint.**

In [ ]:
log.cell_start("Cell 9 — Optional Decoder Fine-Tuning")

import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import random

# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────

FINETUNE_STEPS = 3000
FINETUNE_LR = 2e-4
WARMUP_STEPS = 200
BATCH_SIZE = 4
GRAD_CLIP = 1.0

# Set to False to skip fine-tuning
RUN_FINETUNING = True

if not RUN_FINETUNING:
    print("  ⏭ Skipping decoder fine-tuning (set RUN_FINETUNING=True to enable)")
    log.cell_end("Cell 9 — Optional Decoder Fine-Tuning", "SKIPPED")
else:
    print("\n  Decoder Fine-Tuning on Enriched Field")
    print("  " + "="*50)
    print(f"    Steps: {FINETUNE_STEPS}")
    print(f"    Learning rate: {FINETUNE_LR}")
    print(f"    Batch size: {BATCH_SIZE}")

    # ─────────────────────────────────────────────
    # Prepare training samples from injected data
    # ─────────────────────────────────────────────

    # Combine all datasets
    all_training_texts = []
    for domain, texts in datasets_loaded.items():
        # Sample subset for training (diverse coverage)
        sampled = random.sample(texts, min(1000, len(texts)))
        all_training_texts.extend(sampled)

    random.shuffle(all_training_texts)
    print(f"    Training samples: {len(all_training_texts):,}")

    # ─────────────────────────────────────────────
    # Setup optimizer
    # ─────────────────────────────────────────────

    optimizer = AdamW(model.decoder.parameters(), lr=FINETUNE_LR, weight_decay=0.01)
    scheduler = CosineAnnealingLR(optimizer, T_max=FINETUNE_STEPS, eta_min=1e-6)

    model.decoder.train()
    model.cse.eval()

    # ─────────────────────────────────────────────
    # Training loop
    # ─────────────────────────────────────────────

    losses = []
    best_loss = float('inf')
    start_time = time.time()

    print(f"\n  Starting fine-tuning...")

    for step in range(FINETUNE_STEPS):
        # Sample batch
        batch_texts = random.sample(all_training_texts, min(BATCH_SIZE, len(all_training_texts)))
        
        batch_loss = 0.0
        
        for text in batch_texts:
            if len(text.strip()) < 10:
                continue
            
            try:
                # Encode with CSE
                wave = model.cse.encode(text[:MAX_TEXT_LENGTH])
                wave_seq = wave.full.to(device) if hasattr(wave, 'full') else wave.to(device)
                
                # Get field features at text's location
                coords, _ = text_to_field_coords(text, model.cse, model.field)
                h, w, d = coords
                
                h_slice = slice(max(0, h-2), min(model.field.h, h+2))
                w_slice = slice(max(0, w-2), min(model.field.w, w+2))
                d_slice = slice(max(0, d-2), min(model.field.d, d+2))
                
                field_features = model.field.state[h_slice, w_slice, d_slice].mean(dim=(0,1,2)).to(device)
                
                # Get target bytes
                target_bytes = torch.tensor(
                    [b for b in text[:MAX_TEXT_LENGTH].encode('utf-8')], 
                    dtype=torch.long, 
                    device=device
                )
                
                if len(target_bytes) < 5:
                    continue
                
                # Forward through decoder
                logits = model.decoder(target_bytes, wave_seq, field_features)  # [seq, 256]
                
                # Loss: predict next byte
                loss = F.cross_entropy(
                    logits[:-1].reshape(-1, 256),
                    target_bytes[1:].reshape(-1)
                )
                
                batch_loss += loss
                
            except Exception as e:
                continue
        
        if batch_loss == 0:
            continue
        
        batch_loss = batch_loss / BATCH_SIZE
        
        # Warmup
        if step < WARMUP_STEPS:
            warmup_factor = (step + 1) / WARMUP_STEPS
            for pg in optimizer.param_groups:
                pg['lr'] = FINETUNE_LR * warmup_factor
        
        # Backward
        optimizer.zero_grad()
        batch_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.decoder.parameters(), GRAD_CLIP)
        optimizer.step()
        
        if step >= WARMUP_STEPS:
            scheduler.step()
        
        losses.append(batch_loss.item())
        
        if batch_loss.item() < best_loss:
            best_loss = batch_loss.item()
        
        # Progress
        if (step + 1) % 500 == 0:
            elapsed = time.time() - start_time
            recent_loss = sum(losses[-100:]) / min(100, len(losses))
            eta = (FINETUNE_STEPS - step - 1) / ((step + 1) / elapsed)
            print(f"    Step {step+1:5d}/{FINETUNE_STEPS} | Loss: {recent_loss:.4f} | ETA: {eta/60:.1f}min")

    # ─────────────────────────────────────────────
    # Results
    # ─────────────────────────────────────────────

    model.decoder.eval()
    elapsed = time.time() - start_time

    print(f"\n  " + "="*50)
    print(f"  ✓ Decoder fine-tuning complete!")
    print(f"    Total time: {elapsed/60:.1f} minutes")
    print(f"    Initial loss: {losses[0]:.4f}")
    print(f"    Final loss: {losses[-1]:.4f}")
    print(f"    Best loss: {best_loss:.4f}")

    log.metric("Fine-tune steps", FINETUNE_STEPS)
    log.metric("Final loss", f"{losses[-1]:.4f}")
    log.metric("Training time", f"{elapsed/60:.1f} min")

    log.success("Decoder fine-tuning complete!")
    log.cell_end("Cell 9 — Optional Decoder Fine-Tuning", "PASS")

## Cell 10: Save Enriched Model

In [ ]:
log.cell_start("Cell 10 — Save Enriched Model")

from datetime import datetime

print('\n  Building Flux-X-diverse.flx...')

# Get field dimensions
H, W, D, FEATURES = model.field.h, model.field.w, model.field.d, model.field.features

# ─────────────────────────────────────────────
# Build the .flx archive
# ─────────────────────────────────────────────

flx_diverse = {
    'format': 'FLUX',
    'version': '1.4-diverse',
    
    # Metadata
    'metadata': {
        'source_phase': '9.6',
        'created': datetime.now().isoformat(),
        'description': 'FLUX with diverse dataset injection (training-free physics-based learning)',
        'base_model': str(flx_path.name),
        'injection': {
            'method': 'training_free_physics',
            'datasets': list(datasets_loaded.keys()),
            'total_samples': total_injected,
            'unique_locations': unique_locations,
            'domains': {
                domain: stats['count'] 
                for domain, stats in injection_stats.items()
            },
        },
    },
}

# Copy model components
print('  Extracting model components...')

# CSE
flx_diverse['cse'] = {
    'config': {'wave_dim': 432},
    'state_dict': model.cse.state_dict(),
}
print('    ✓ CSE')

# Field (enriched with diverse knowledge!)
flx_diverse['field'] = {
    'config': {'h': H, 'w': W, 'd': D, 'features': FEATURES},
    'state_dict': model.field.state_dict(),
}
if hasattr(model, 'gr'):
    flx_diverse['field']['gravity_state'] = model.gr.save_state()
if hasattr(model, 'tl'):
    flx_diverse['field']['thermodynamic_state'] = model.tl.save_state()
print('    ✓ Field (ENRICHED with diverse knowledge)')

# Memory
flx_diverse['memory'] = {}
if hasattr(model, 'working_memory'):
    flx_diverse['memory']['working'] = model.working_memory.state_dict_memory()
if hasattr(model, 'episodic_memory'):
    flx_diverse['memory']['episodic'] = model.episodic_memory.save_state()
if hasattr(model, 'semantic_memory'):
    flx_diverse['memory']['semantic'] = model.semantic_memory.save_state()
print('    ✓ Memory')

# Decoder (possibly fine-tuned)
decoder_state = model.decoder.state_dict()
cleaned_decoder = {k.replace('_orig_mod.', ''): v for k, v in decoder_state.items()}
flx_diverse['decoder'] = {
    'config': {'hidden_dim': 1024, 'num_layers': 4},
    'state_dict': cleaned_decoder,
}
print('    ✓ WaveDecoder')

# Causal
flx_diverse['causal'] = {}
if hasattr(model, 'cgn'):
    flx_diverse['causal']['cgn_state'] = model.cgn.save_state()
if hasattr(model, 'causal_graph'):
    flx_diverse['causal']['graph_state'] = model.causal_graph.save_state()
print('    ✓ CGN + CausalGraph')

# Bridges
flx_diverse['bridges'] = {}
if hasattr(model, 'wave_to_field'):
    flx_diverse['bridges']['wave_to_field'] = model.wave_to_field.state_dict()
if hasattr(model, 'field_to_wave'):
    flx_diverse['bridges']['field_to_wave'] = model.field_to_wave.state_dict()
if hasattr(model, 'memory_router'):
    flx_diverse['bridges']['router'] = model.memory_router.save_state()
if hasattr(model, 'output_head'):
    flx_diverse['bridges']['output_head'] = model.output_head.state_dict()
print('    ✓ Bridges')

# Include adapters if they exist
original_flx = torch.load(str(flx_path), map_location='cpu', weights_only=False)
if 'adapters' in original_flx:
    flx_diverse['adapters'] = original_flx['adapters']
    print('    ✓ Adapters (copied from original)')

# ─────────────────────────────────────────────
# Save locally
# ─────────────────────────────────────────────

output_path = CHECKPOINT_DIR / 'Flux-X-diverse.flx'
torch.save(flx_diverse, str(output_path))
size_mb = output_path.stat().st_size / 1e6
print(f'\n  ✓ Saved: {output_path.name} ({size_mb:.1f} MB)')

log.metric("Output file", output_path.name)
log.metric("File size", f"{size_mb:.1f} MB")

log.cell_end("Cell 10 — Save Enriched Model", "PASS")

## Cell 11: Upload to HuggingFace Hub

In [ ]:
log.cell_start("Cell 11 — Upload to HuggingFace")

print('\n  Uploading to HuggingFace Hub...')

from huggingface_hub import HfApi

if hf_token:
    try:
        api = HfApi(token=hf_token)
        
        api.upload_file(
            path_or_fileobj=str(output_path),
            path_in_repo='checkpoints/Flux-X-diverse.flx',
            repo_id='UnseenGAP/FLUX',
            commit_message=f'Add Flux-X-diverse.flx (Phase 9.6 diverse dataset injection) — {datetime.now().isoformat()}',
        )
        
        print(f'  ✓ Uploaded to HuggingFace Hub')
        print(f'    https://huggingface.co/UnseenGAP/FLUX/blob/main/checkpoints/Flux-X-diverse.flx')
        log.success("Flux-X-diverse.flx uploaded to HuggingFace")
        
    except Exception as e:
        print(f'  ⚠ Upload failed: {e}')
        print(f'    File saved locally: {output_path}')
        log.warning(f"HF upload failed: {str(e)[:50]}")
else:
    log.warning("No HF_TOKEN - skipping upload")
    log.info(f"Model saved locally at: {output_path}")

log.cell_end("Cell 11 — Upload to HuggingFace", "PASS")

## Cell 12: Summary & Next Steps

In [ ]:
log.separator("Phase 9.6 Complete: Diverse Dataset Injection")

print("""
╔════════════════════════════════════════════════════════════════════╗
║                    PHASE 9.6 COMPLETE                             ║
║           TRAINING-FREE DIVERSE DATASET INJECTION                  ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  WHAT WE DID (NO BACKPROPAGATION):                                ║
║  1. Loaded 5 diverse datasets:                                     ║
║     • OpenAssistant (helpful responses)                           ║
║     • Dolly-15k (natural dialogue)                                ║
║     • GSM8K (math reasoning)                                       ║
║     • Capybara (multi-turn conversations)                         ║
║     • CodeAlpaca (programming)                                     ║
║                                                                    ║
║  2. For each text sample:                                          ║
║     • CSE.encode() → deterministic wave                           ║
║     • hash + wave → field coordinates                             ║
║     • field.state[coords] += wave (knowledge injection)           ║
║     • field.mass[coords] += 1.0 (evidence accumulation)           ║
║                                                                    ║
║  3. Thermodynamic settling every 500 samples                      ║
║     • Diffusion smoothing                                          ║
║     • Energy minimization                                          ║
║     • Natural clustering of related concepts                      ║
║                                                                    ║
║  THIS IS FLUX PHYSICS:                                             ║
║  ├── No loss functions                                            ║
║  ├── No gradients                                                  ║
║  ├── No epochs                                                     ║
║  └── Just physics: waves, fields, mass, energy                    ║
║                                                                    ║
║  OUTPUT: Flux-X-diverse.flx                                        ║
║  • Contains: assistancy, dialogue, reasoning, code knowledge      ║
║  • Field topology now reflects diverse human knowledge             ║
║  • Decoder fine-tuning optional (Cell 9)                          ║
║                                                                    ║
╚════════════════════════════════════════════════════════════════════╝
""")

# Print actual injection stats
print("\nINJECTION STATISTICS:")
print(f"  Total samples injected: {total_injected:,}")
print(f"  Unique field locations: {unique_locations:,}")
print(f"  Coverage: {100*unique_locations/(H*W*D):.2f}% of field")
print(f"  Injection time: {total_time:.1f}s")
print()

print("DOMAINS INJECTED:")
for domain, stats in injection_stats.items():
    print(f"  {domain:12s}: {stats['count']:,} samples")
print()

print("NEXT STEPS:")
print("  1. Run this notebook on Kaggle/Colab with GPU")
print("  2. Test generation quality")
print("  3. If needed, run Cell 9 for decoder fine-tuning")
print("  4. Upload Flux-X-diverse.flx to HuggingFace")
print()
print("  The field now contains DIVERSE human knowledge:")
print("  • How to be helpful (assistancy)")
print("  • How to converse naturally (dialogue)")
print("  • How to reason step-by-step (reasoning)")
print("  • How to write code (programming)")

log.success("Phase 9.6 notebook complete!")